In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Install FFmpeg and extract audios

In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,564 kB]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,633 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,858 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,966 kB]
Get:14 http://securit

In [ ]:
import os
from pathlib import Path

video_dir = Path("/content/drive/MyDrive/job_recommendation_dataset/videos")
audio_dir = Path("/content/drive/MyDrive/job_recommendation_dataset/extracted_audios")

video_files = list(video_dir.glob("*.mp4"))

In [ ]:
import subprocess

for video in video_files:
    out_wav = audio_dir / (video.stem + ".wav")
    subprocess.run(["ffmpeg" , "-y" , "-i", str(video) , "-vn" , "ac" , "1" , "-ar" , "16000",str(out_wav)])

In [ ]:
audio_dir = Path("/content/drive/MyDrive/job_recommendation_dataset/extracted_audios")
wav_files = list(audio_dir.glob("*.wav"))

In [ ]:
!pip -q install librosa

In [ ]:
import numpy as np
import pandas as pd
import librosa

# Getting Transcript for words per minute feature

In [ ]:
trans_dir = Path("/content/drive/MyDrive/job_recommendation_dataset/extracted_transcripts")

def get_transcript(wav_path):
    wav_path = Path(wav_path)
    return (trans_dir / f"{wav_path.stem}.txt").read_text()

## Features To extract from audio:
- **Duration of video**
- **Loudness (voice energy) from RMS (Root Mean Square Energy)**
- **Noisiness from ZCR (Zero Crossing Rate)**
- **brightness from spectral_centroid**
- **Silence detection**
- **Long pauses & longer than 0.5 sec**
- **Speaking rate & count of words per min**



- **Vocal energy and presence** (through loudness)
- **Speech clarity and fluency** (through pause analysis)
- **Vocal tone and expressiveness** (through spectral features)
- **Speech pace and rhythm** (through timing metrics)

In [ ]:
def extract_audio_features(wav_path, transcript_text=None):
    audio, sample_rate = librosa.load(wav_path, sr=16000, mono=True)
    duration = len(audio) / sample_rate

    def find_pauses(is_silent, hop_length, sample_rate):
      pauses = []
      pause_start = None
      frame_duration = hop_length / sample_rate

      for i, silent in enumerate(is_silent):
        if silent and pause_start is None:
          pause_start = i
        elif not silent and pause_start is not None:
          pause_length = (i - pause_start) * frame_duration
          pauses.append(pause_length)
          pause_start = None

      return pauses

    loudness = librosa.feature.rms(y=audio)[0]
    noisiness = librosa.feature.zero_crossing_rate(audio)[0]
    brightness = librosa.feature.spectral_centroid(y=audio, sr=sample_rate)[0]

    # Detect silence (frames below 20th percentile of loudness)
    silence_threshold = np.percentile(loudness, 20)
    is_silent = loudness < silence_threshold

    # Calculate pauses
    pauses = find_pauses(is_silent, hop_length=512, sample_rate=sample_rate)
    long_pauses = [p for p in pauses if p >= 0.5]

    # Calculate speaking rate
    words_per_min = None
    if transcript_text:
        word_count = len(transcript_text.strip().split())
        words_per_min = (word_count / duration) * 60

    return {
        "filename": Path(wav_path).stem,
        "duration_sec": round(duration, 2),

        # Loudness
        "loudness_mean": round(float(loudness.mean()), 4),
        "loudness_std": round(float(loudness.std()), 4),

        # Noisiness (zero crossing rate)
        "noisiness_mean": round(float(noisiness.mean()), 4),
        "noisiness_std": round(float(noisiness.std()), 4),

        # Brightness (spectral centroid)
        "brightness_mean": round(float(brightness.mean()), 2),
        "brightness_std": round(float(brightness.std()), 2),

        # Silence and speech
        "silence_ratio": round(float(is_silent.mean()), 3),
        "speech_ratio": round(1.0 - float(is_silent.mean()), 3),
        "pause_count": len(pauses),
        "long_pause_count": len(long_pauses),
        "long_pause_total_sec": round(sum(long_pauses), 2),
        "long_pause_max_sec": round(max(long_pauses), 2) if long_pauses else 0.0,

        # Speaking rate
        "words_per_min": round(words_per_min, 1) if words_per_min else None,
    }

In [ ]:
rows = []
for w in wav_files:
      transcript = get_transcript(w)
      rows.append(extract_audio_features(w , transcript_text=transcript))

df = pd.DataFrame(rows)
df

,filename,duration_sec,loudness_mean,loudness_std,noisiness_mean,noisiness_std,brightness_mean,brightness_std,silence_ratio,speech_ratio,pause_count,long_pause_count,long_pause_total_sec,long_pause_max_sec,words_per_min
0,1,40.75,0.0292,0.0228,0.1737,0.1187,1889.38,1003.34,0.200,0.800,39,6,3.39,0.67,153.1
1,13,55.32,0.0696,0.0464,0.0932,0.0718,1149.93,780.30,0.200,0.800,40,2,4.16,3.62,165.9
2,11,87.53,0.0447,0.0295,0.1239,0.1128,1529.17,1099.11,0.200,0.800,68,12,8.67,1.28,154.2
3,10,72.90,0.0421,0.0275,0.1178,0.1136,1524.25,983.53,0.200,0.800,36,2,2.88,2.30,156.4
4,12,75.93,0.0686,0.0425,0.1146,0.0990,1476.16,950.45,0.200,0.800,54,5,5.63,3.46,169.1
5,20,40.40,0.0569,0.0329,0.1578,0.0786,2016.40,772.22,0.200,0.800,78,0,0.00,0.00,212.4
6,18,13.35,0.1173,0.0545,0.1649,0.1356,2032.16,1140.14,0.201,0.799,17,1,0.64,0.64,134.8
7,2,38.52,0.0301,0.0271,0.1635,0.0986,1850.01,902.64,0.200,0.800,29,5,3.87,0.99,193.1
8,21,19.78,0.0734,0.0457,0.1330,0.0941,1620.01,881.55,0.200,0.800,19,3,2.21,0.86,221.4
9,19,27.54,0.1142,0.0537,0.1520,0.0858,1980.83,814.56,0.200,0.800,43,0,0.00,0.00,180.8


In [ ]:
out_path = "/content/drive/MyDrive/job_recommendation_dataset/audioExtractedFeatures.csv"
df.to_csv(out_path, index=False)

out_path

'/content/drive/MyDrive/job_recommendation_dataset/audioExtractedFeatures.csv'